<a href="https://colab.research.google.com/github/SridharS-Square/Agentic_AI_Workshop/blob/main/Building%20your%20First%20Al%20Agent%20with%20AutoGen/Streamlining_Exploratory_Data_Analysis_(EDA)_with_a_Multi_Agent_System_using_Autogen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyautogen pandas matplotlib scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 4.4 MB/s eta 0:00:00


In [4]:
import os
from google.colab import userdata
import pyautogen as autogen # Changed import to pyautogen

gemini_api_key = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = gemini_api_key

# Update the config_list to use Gemini 1.5 Flash
config_list = [
    {
        "model": "gemini-1.5-flash-latest", # Switched to the Flash model
        "api_key": gemini_api_key,
        "api_type": "google"
    }
]

# Common LLM configuration for all agents
llm_config = {
    "config_list": config_list,
    "cache_seed": 42, # Use a seed for caching to get consistent results
}

print("✅ AutoGen is now configured to use the Gemini 1.5 Flash API.")

✅ AutoGen is now configured to use the Gemini 1.5 Flash API.


In [ ]:
# Create the Admin/Executor Agent
import pyautogen as autogen # Added import

from autogen.agentchat import UserProxyAgent, ConversableAgent

user_proxy = UserProxyAgent(
   name="Admin_Executor",
   system_message="A human admin who is the project lead. Interacts with the team to check on progress and kickstarts the workflow. Also executes code provided by other agents.",
   code_execution_config={"work_dir": "eda_work_dir"}, # Sets the working directory for code execution
   human_input_mode="NEVER", # Fully automated
   llm_config=llm_config, # Added llm_config back
)

# Create the Data Preparation Agent 🧹
data_prep_agent = ConversableAgent(
    name="Data_Preparation_Agent",
    system_message='''You are the Data Preparation Agent. Your job is to load the dataset from the provided file path ('iris_dataset.csv').
    You must perform initial data cleaning and preprocessing. This includes:
    1.  Checking for and handling any missing values.
    2.  Verifying data types are correct.
    3.  Providing a summary of the cleaning process.
    Do not perform any analysis or visualization. Just prepare the data for the EDA Agent.
    ''',
    llm_config=llm_config, # Added llm_config back
)

# Create the EDA Agent 📊
eda_agent = ConversableAgent(
    name="EDA_Agent",
    system_message='''You are the EDA Agent. You are an expert in statistical analysis and data visualization.
    After the data is prepared, you must:
    1.  Perform a detailed statistical summary of the dataset (e.g., using describe()).
    2.  Generate at least two insightful visualizations (e.g., a histogram of a feature and a scatter plot of two features).
    3.  **Important**: You must save the visualizations as image files (e.g., 'histogram.png', 'scatterplot.png') in the working directory.
    4.  Summarize your key findings from the analysis.
    ''',
    llm_config=llm_config, # Added llm_config back
)

# Create the Report Generator Agent 📝
report_agent = ConversableAgent(
    name="Report_Generator_Agent",
    system_message='''You are the Report Generator Agent. Your task is to compile a final, well-structured EDA report in markdown format.
    You must gather all the information from the Data Preparation Agent and the EDA Agent, including:
    1.  The data cleaning summary.
    2.  The statistical summary.
    3.  Key insights from the analysis.
    4.  **Crucially, you must include the generated visualizations in your markdown report using markdown syntax for images (e.g., ![Histogram](histogram.png)).**
    Structure the report logically with clear headings.
    ''',
    llm_config=llm_config, # Added llm_config back
)

# Create the Critic Agent 🕵️
critic_agent = ConversableAgent(
    name="Critic_Agent",
    system_message='''You are the Critic Agent. Your role is to review the final EDA report.
    You must check for:
    1.  **Clarity**: Is the report easy to understand?
    2.  **Accuracy**: Are the statistical summaries and findings correct?
    3.  **Completeness**: Does the report include all the required components (data cleaning, stats, visualizations)?
    Provide constructive feedback to the team on how to improve the report. If the report is satisfactory, you will give your final approval.
    ''',
    llm_config=llm_config, # Added llm_config back
)

print("✅ All agents have been defined.")

In [ ]:
# Create the Group Chat 🤝
groupchat = autogen.GroupChat(
    agents=[user_proxy, data_prep_agent, eda_agent, report_agent, critic_agent],
    messages=[],
    max_round=15  # Set a limit to the number of conversation turns
)

# Create the Group Chat Manager
manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

print("✅ Group Chat and Manager have been created.")

In [ ]:
# The initial task for the team
task = """
Perform a comprehensive Exploratory Data Analysis (EDA) on the 'iris_dataset.csv' file.

Your process must follow these steps:
1. The Data Preparation Agent must first clean the data.
2. The EDA Agent must then perform statistical analysis and create visualizations, saving them as files.
3. The Report Generator Agent must compile all findings into a structured markdown report.
4. The Critic Agent must review the report and provide feedback. The team should refine the report based on this feedback until the Critic gives its final approval.

Begin the workflow now.
"""

# Initiate the chat
user_proxy.initiate_chat(
    manager,
    message=task
)